# GSW Transformation of NetCDF CTD Data (TEOS-10)

# GSW Transformation of CTD Data

This notebook processes NetCDF CTD files and converts:

- Practical Salinity → Absolute Salinity (SA)
- In-situ Temperature → Conservative Temperature (CT)

using the TEOS-10 Gibbs SeaWater (GSW) toolbox.

### Key Features:
- Handles missing or invalid pressure
- Computes pressure from depth if needed
- Processes multiple files from a predefined list
- Outputs new NetCDF files
- Generates a processing summary table + CSV

In [1]:
import os
import netCDF4 as nc
import numpy as np
import gsw
import warnings
import pandas as pd
from prettytable import PrettyTable

warnings.filterwarnings("ignore", category=UserWarning)

## Helper Functions

These functions handle:
- Reading file lists
- Computing pressure (fallback to calculating pressure from depth if needed)

In [2]:
def read_filenames_from_txt(txt_filepath):
    with open(txt_filepath, 'r') as file:
        return [line.strip() for line in file.readlines()]


def compute_pressure(ds):
    """Return pressure array if valid; otherwise compute from depth."""

    if "pressure" in ds.variables:
        p = np.ma.filled(ds.variables["pressure"][:], np.nan)
        if np.isfinite(p).any():
            return p, "pressure"
        else:
            print("Warning: Pressure exists but is fully masked; using depth instead")
    else:
        print("Warning: No pressure variable found; using depth instead")

    if "depth" not in ds.variables:
        raise RuntimeError("Neither 'pressure' nor 'depth' found in file")

    depth = np.ma.filled(ds.variables["depth"][:], np.nan)
    lat = np.ma.filled(ds.variables["lat"][:], np.nan)

    if not np.isfinite(lat).any():
        raise RuntimeError("Latitude invalid — cannot compute pressure from depth")

    lat_mean = np.nanmean(lat)
    p = gsw.p_from_z(-depth, lat_mean)

    return p, "depth (computed)"

## Main Processing Function

This function:
- Filters selected NetCDF files
- Applies GSW transformations
- Writes output files
- Tracks processing results

In [3]:
def GSW_transform_nc_files(input_dir, txt_filepath, output_dir):

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    selected_files = read_filenames_from_txt(txt_filepath)
    nc_files = [f for f in os.listdir(input_dir) if f.endswith(".nc") and f in selected_files]

    summary = []

    for filename in nc_files:

        input_path = os.path.join(input_dir, filename)
        new_filename = f"{os.path.splitext(filename)[0]}_gsw.nc"
        output_path = os.path.join(output_dir, new_filename)

        if os.path.exists(output_path):
            print(f">> Skipping {filename} (already processed)")
            summary.append((filename, "skipped", "-", "-", "-"))
            continue

        print(f"\nProcessing {filename} ...")

        try:
            with nc.Dataset(input_path, "r") as ds:

                temperature = np.ma.filled(ds.variables["temperature"][:], np.nan)
                salinity = np.ma.filled(ds.variables["salinity"][:], np.nan)
                lat = np.ma.filled(ds.variables["lat"][:], np.nan)
                lon = np.ma.filled(ds.variables["lon"][:], np.nan)

                pressure, pressure_source = compute_pressure(ds)

                # Broadcast lat/lon
                while lat.ndim < salinity.ndim:
                    lat = np.expand_dims(lat, axis=-1)
                while lon.ndim < salinity.ndim:
                    lon = np.expand_dims(lon, axis=-1)

                lat = np.broadcast_to(lat, salinity.shape)
                lon = np.broadcast_to(lon, salinity.shape)

                abs_salinity = gsw.SA_from_SP(salinity, pressure, lon, lat)
                cons_temperature = gsw.CT_from_t(abs_salinity, temperature, pressure)

                abs_salinity = np.ma.masked_invalid(abs_salinity)
                cons_temperature = np.ma.masked_invalid(cons_temperature)

                sa_valid = np.isfinite(abs_salinity).any()
                ct_valid = np.isfinite(cons_temperature).any()

                # Write output
                with nc.Dataset(output_path, "w", format="NETCDF4_CLASSIC") as ds_out:

                    for name, dim in ds.dimensions.items():
                        ds_out.createDimension(name, len(dim) if not dim.isunlimited() else None)

                    for name, var in ds.variables.items():
                        if name in ("temperature", "salinity"):
                            continue

                        out_var = ds_out.createVariable(name, var.datatype, var.dimensions)
                        attrs = {a: var.getncattr(a) for a in var.ncattrs() if a != "_FillValue"}
                        out_var.setncatts(attrs)
                        out_var[:] = var[:]

                    sa_var = ds_out.createVariable("absolute_salinity", "f4", ds["salinity"].dimensions, fill_value=np.nan)
                    ct_var = ds_out.createVariable("conservative_temperature", "f4", ds["temperature"].dimensions, fill_value=np.nan)

                    sa_var.units = "g/kg"
                    sa_var.long_name = "Absolute Salinity (TEOS-10)"
                    sa_var.standard_name = "sea_water_absolute_salinity"

                    ct_var.units = "degC"
                    ct_var.long_name = "Conservative Temperature (TEOS-10)"
                    ct_var.standard_name = "sea_water_conservative_temperature"

                    sa_var[:] = abs_salinity
                    ct_var[:] = cons_temperature

                summary.append((filename, "processed", pressure_source, sa_valid, ct_valid))
                print(f"Saved: {new_filename}")

        except Exception as e:
            summary.append((filename, f"error: {e}", "-", "-", "-"))
            print(f"Error processing {filename}: {e}")

    # Summary table
    print("\nProcessing Summary")
    table = PrettyTable()
    table.field_names = ["Filename", "Status", "Pressure Source", "SA valid", "CT valid"]

    for row in summary:
        table.add_row(row)

    print(table)

    # Export CSV
    csv_path = os.path.join(output_dir, "GSW_processing_summary.csv")
    df_summary = pd.DataFrame(summary, columns=["Filename", "Status", "Pressure Source", "SA valid", "CT valid"])
    df_summary.to_csv(csv_path, index=False)

    print(f"\nSummary exported to: {csv_path}")

## Input / Output Configuration

Set from config.yaml

In [ ]:
# Define file paths from config file 
from config_loader import load_paths

# Load all base paths
paths = load_paths()

input_dir = paths["omg_dir"]
output_dir = paths["nc_dir"]
csv_dir = paths["csv_dir"]


# Script-specific txt file
txt_filepath = csv_dir / "allFilenames.txt"


## Run script



In [5]:
GSW_transform_nc_files(input_dir, txt_filepath, output_dir)



Processing OMG_Ocean_AXCTD_L2_20160915151827.nc ...
Saved: OMG_Ocean_AXCTD_L2_20160915151827_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20160915161939.nc ...
Saved: OMG_Ocean_AXCTD_L2_20160915161939_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20160915164446.nc ...
Saved: OMG_Ocean_AXCTD_L2_20160915164446_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20160923151432.nc ...
Saved: OMG_Ocean_AXCTD_L2_20160923151432_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20160923152021.nc ...
Saved: OMG_Ocean_AXCTD_L2_20160923152021_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20160924173732.nc ...
Saved: OMG_Ocean_AXCTD_L2_20160924173732_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20160924175043.nc ...
Saved: OMG_Ocean_AXCTD_L2_20160924175043_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20180824120642.nc ...
Saved: OMG_Ocean_AXCTD_L2_20180824120642_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20180824122629.nc ...
Saved: OMG_Ocean_AXCTD_L2_20180824122629_gsw.nc

Processing OMG_Ocean_AXCTD_L2_20180825122858.nc ...
Saved: OMG_Ocean_AXCTD_L2_201808251228